# 04 — Frozen-residual anomaly models

This notebook implements one clear sequence:

1. measurement-kind transformations;
2. a calibration-frozen robust reference;
3. rapid and drift statistical channels;
4. independent per-channel threshold calibration;
5. comparison at equal **case** workload on development data.

The holdout is not opened here.


## 1. Setup


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")  # or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run2",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
EVAL_ROOT = RUN_ROOT / "SPEC-EVAL"
SPLIT_ROOT = RUN_ROOT / "SPLITS"

import tempfile
import time

import duckdb
import joblib

import evaluation_core
import simple_model_core
evaluation_core = importlib.reload(evaluation_core)
simple_model_core = importlib.reload(simple_model_core)
from evaluation_core import evaluate_cases, form_cases
from simple_model_core import (
    MODEL_CORE_VERSION, alert_grid_from_score_file, case_score_trace,
    calibration_thresholds, fit_residual_bundle,
    duration_to_observations,
    materialize_measurement_features, materialize_wide_partition,
    partition_exposure, score_residual_file,
)

EDA_VERSION = "2.1.0"
EVALUATION_VERSION = MODEL_VERSION = "2.2.0"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / f"{SECTOR}_eda_v2_1_run1"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{EVALUATION_VERSION}" / SECTOR / f"{SECTOR}_evaluation_v2_2_run1"
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{MODEL_VERSION}" / SECTOR / f"{SECTOR}_models_v2_2_run1"
LEGACY_ROOT = (
    DATA_ROOT / "outputs" / "models" / "v2.0.0" / SECTOR
    / f"{SECTOR}_models_v2_run1"
)
MAX_TRAINING_ROWS = int(os.getenv("MODEL_MAX_TRAINING_ROWS", "150000"))

manifest = read_json(CORE_ROOT / "manifest.json")
decisions = read_json(EDA_ROOT / "eda_decisions.json")
policy = read_json(EVALUATION_ROOT / "evaluation_policy.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
fault_events = pd.read_parquet(EVALUATION_ROOT / "development" / "fault_events.parquet")
fault_intervals = pd.read_parquet(EVALUATION_ROOT / "development" / "fault_entity_intervals.parquet")
baseline_review = pd.read_parquet(EDA_ROOT / "baseline_review.parquet")
group_path = SPLIT_ROOT / "entity_groups.parquet"
entity_groups = pd.read_parquet(group_path) if group_path.is_file() else pd.DataFrame()

if decisions["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("EDA and canonical input do not match")
if policy["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("Evaluation and canonical input do not match")

display(pd.Series({
    "sector": SECTOR,
    "reference": "frozen calibration median / robust scale",
    "training_rows_cap": MAX_TRAINING_ROWS,
    "selection_channels": policy["selection_channels"],
    "output": str(MODEL_ROOT),
}, name="value").to_frame())


## 2. Transform, fit on calibration and score development


In [ ]:
started = time.perf_counter()
temporary = tempfile.TemporaryDirectory()
work = Path(temporary.name)
paths = {}
lookback_seconds = max(
    decisions["dispersion_window_seconds"],
    decisions["base_cadence_seconds"],
)

for partition in ("calibration", "development"):
    wide = work / f"{partition}_wide.parquet"
    feature_path = work / f"{partition}_features.parquet"
    split = materialize_wide_partition(
        CORE_ROOT, SPLIT_ROOT, partition, catalogue, wide,
        lookback_seconds=lookback_seconds,
    )
    if partition == "calibration":
        materialize_measurement_features(
            wide, catalogue, feature_path,
            score_start=split["score_start"], score_end=split["score_end"],
        )
    else:
        materialize_measurement_features(wide, catalogue, feature_path)
    paths[partition] = {"wide": wide, "features": feature_path, **split}

bundle = fit_residual_bundle(
    paths["calibration"]["features"],
    use_entity_reference=decisions["primary_split"] == "time",
    reference_exclusions=baseline_review.loc[
        baseline_review.review_flag, ["entity_id", "metric_id"]
    ],
    maximum_training_rows=MAX_TRAINING_ROWS,
    fit_multivariate=False,
)
for partition in paths:
    score_path = work / f"{partition}_scores.parquet"
    score_residual_file(
        bundle, paths[partition]["features"], score_path,
        cadence_seconds=decisions["base_cadence_seconds"],
        dispersion_window_seconds=decisions["dispersion_window_seconds"],
        cusum_allowance=policy["cusum_allowance"],
        score_start=paths[partition]["score_start"],
        score_end=paths[partition]["score_end"],
    )
    paths[partition]["scores"] = score_path

display(pd.Series({
    "fitted_features": len(bundle["feature_columns"]),
    "entity_specific_reference": bundle["use_entity_reference"],
    "pooled_reference_fallbacks": len(bundle["reference_exclusions"]),
    "lookback_seconds": lookback_seconds,
    "elapsed_minutes": (time.perf_counter() - started) / 60,
}, name="value").to_frame())


## 3. Calibration thresholds and readiness


In [ ]:
block_column = "entity_id" if decisions["primary_split"] == "time" else "episode_id"
thresholds = calibration_thresholds(
    paths["calibration"]["scores"],
    policy["threshold_quantiles"],
    block_column=block_column,
    minimum_block_rows=20,
    model_ids=policy["selection_channels"],
)

with duckdb.connect() as connection:
    score_file = str(paths["development"]["scores"])
    readiness_base = connection.execute("""
        SELECT entity_id, readiness AS status, count(*) AS observations
        FROM read_parquet(?)
        GROUP BY entity_id, readiness
        ORDER BY entity_id, readiness
    """, [score_file]).df()
readiness = pd.concat([
    readiness_base.assign(channel=channel)
    for channel in policy["selection_channels"]
], ignore_index=True)[["entity_id", "channel", "status", "observations"]]
display(thresholds)
display(readiness.groupby(["channel", "status"]).observations.sum().to_frame())


## 4. Compare the two predeclared portfolios

Rapid and drift are the two channels justified by the current diagnosis.
Their thresholds vary independently on a predeclared grid. Extra channels
must demonstrate incremental value in a later, separately evaluated step.


In [ ]:
PORTFOLIOS = {
    "rapid_only": ["rapid_residual"],
    "rapid_and_drift": ["rapid_residual", "drift_cusum"],
}
PREFERENCE = {name: rank for rank, name in enumerate(PORTFOLIOS)}
exposure = partition_exposure(
    paths["development"]["scores"], policy["exposure_unit"],
    decisions["base_cadence_seconds"],
)

cadence_seconds = decisions["base_cadence_seconds"]
persistence_observations = {
    channel: duration_to_observations(seconds, cadence_seconds)
    for channel, seconds in policy["channel_persistence_seconds"].items()
}
recovery_observations = duration_to_observations(
    policy["recovery_seconds"], cadence_seconds
)
missing_time_rules = set(policy["selection_channels"]) - set(persistence_observations)
if missing_time_rules:
    raise ValueError(f"Missing persistence durations: {sorted(missing_time_rules)}")
time_policy = pd.DataFrame({
    "channel": list(persistence_observations),
    "persistence_seconds": [
        policy["channel_persistence_seconds"][channel]
        for channel in persistence_observations
    ],
    "persistence_observations": list(persistence_observations.values()),
})
display(time_policy)
print(f"Recovery: {policy['recovery_seconds']} seconds = {recovery_observations} observations")
print(f"Case gap: {policy['case_gap_seconds']} seconds")

def threshold_value(channel, quantile):
    row = thresholds.loc[
        thresholds.model_id.eq(channel)
        & thresholds.threshold_quantile.eq(float(quantile)),
        "threshold",
    ]
    if len(row) != 1:
        raise ValueError(f"Missing threshold for {channel} at {quantile}")
    return float(row.iloc[0])

alert_grid = alert_grid_from_score_file(
    paths["development"]["scores"], thresholds,
    persistence=persistence_observations,
    recovery_consecutive=recovery_observations,
)

def make_alerts(channels, channel_quantiles):
    channel_thresholds = {
        channel: threshold_value(channel, channel_quantiles[channel])
        for channel in channels
    }
    frames = [
        alert_grid[(channel, float(channel_quantiles[channel]))]
        for channel in channels
    ]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        from evaluation_core import ALERT_COLUMNS
        return pd.DataFrame(columns=ALERT_COLUMNS), channel_thresholds
    alerts = pd.concat(frames, ignore_index=True).sort_values("alert_start").reset_index(drop=True)
    alerts["alert_id"] = [f"A-{number:09d}" for number in range(1, len(alerts) + 1)]
    return alerts, channel_thresholds

quantiles = policy["threshold_quantiles"]
candidate_specs = [
    ("rapid_only", {"rapid_residual": quantile})
    for quantile in quantiles
] + [
    ("rapid_and_drift", {"rapid_residual": rapid, "drift_cusum": drift})
    for rapid in quantiles for drift in quantiles
]

comparisons = []
for portfolio, channel_quantiles in candidate_specs:
    channels = PORTFOLIOS[portfolio]
    alerts, channel_thresholds = make_alerts(channels, channel_quantiles)
    cases, members = form_cases(
        alerts, entity_groups,
        gap_seconds=policy["case_gap_seconds"],
        thresholds=channel_thresholds,
    )
    result = evaluate_cases(
        cases, members, fault_events, fault_intervals,
        exposure_value=exposure,
        exposure_unit=policy["exposure_unit"],
        decision_horizon_seconds=policy["decision_horizon_seconds"],
    )
    metric_table = result["metrics"].set_index("metric")
    false_name = f"false_cases_per_{policy['exposure_unit']}"
    total_name = f"total_cases_per_{policy['exposure_unit']}"
    setting = ", ".join(
        f"{channel}={channel_quantiles[channel]}" for channel in channels
    )
    key = f"{portfolio}|{setting}"
    comparisons.append({
        "candidate_key": key,
        "portfolio": portfolio,
        "threshold_setting": setting,
        "rapid_quantile": channel_quantiles["rapid_residual"],
        "drift_quantile": channel_quantiles.get("drift_cusum", np.nan),
        "channels": ", ".join(channels),
        "event_recall": metric_table.at["event_recall", "value"],
        "event_recall_ci_low": metric_table.at["event_recall", "ci_low"],
        "event_recall_ci_high": metric_table.at["event_recall", "ci_high"],
        "preimpact_event_recall": metric_table.at["preimpact_event_recall", "value"],
        "case_precision": metric_table.at["case_precision", "value"],
        "false_case_rate": metric_table.at[false_name, "value"],
        "false_case_rate_ci_low": metric_table.at[false_name, "ci_low"],
        "false_case_rate_ci_high": metric_table.at[false_name, "ci_high"],
        "total_case_rate": metric_table.at[total_name, "value"],
        "median_delay_seconds": metric_table.at["median_detection_delay_seconds", "value"],
        "cases": len(cases),
    })
    print(f"Evaluated {portfolio}, {setting}: {len(cases):,} cases")

comparison = pd.DataFrame(comparisons)
display(comparison.sort_values(["portfolio", "rapid_quantile", "drift_quantile"]))


## 5. Apply the predeclared selection rule


In [ ]:
budget_gate = policy["false_case_budget"] * policy["budget_safety_factor"]
eligible = comparison.loc[
    comparison.false_case_rate_ci_high.le(budget_gate)
].copy()
selection_status = "within_budget"
if eligible.empty:
    eligible = comparison.nsmallest(1, "false_case_rate_ci_high").copy()
    selection_status = "no_configuration_within_budget"

best_recall = eligible.event_recall.max()
equivalent = eligible.loc[
    eligible.event_recall.ge(best_recall - policy["equivalence_margin"])
].copy()
equivalent["preference"] = equivalent.portfolio.map(PREFERENCE)
selected = equivalent.sort_values(
    ["preference", "false_case_rate_ci_high", "median_delay_seconds", "threshold_setting"],
    ascending=[True, True, True, True],
).iloc[0]

selected_quantiles = {"rapid_residual": float(selected.rapid_quantile)}
if selected.portfolio == "rapid_and_drift":
    selected_quantiles["drift_cusum"] = float(selected.drift_quantile)
selected_channels = PORTFOLIOS[selected.portfolio]
alerts, selected_thresholds = make_alerts(selected_channels, selected_quantiles)
cases, members = form_cases(
    alerts, entity_groups, gap_seconds=policy["case_gap_seconds"],
    thresholds=selected_thresholds,
)
result = evaluate_cases(
    cases, members, fault_events, fault_intervals,
    exposure_value=exposure, exposure_unit=policy["exposure_unit"],
    decision_horizon_seconds=policy["decision_horizon_seconds"],
)

legacy_summary = pd.DataFrame()
legacy_fault_comparison = pd.DataFrame()
legacy_metrics_path = LEGACY_ROOT / "development_metrics.csv"
legacy_faults_path = LEGACY_ROOT / "fault_type_results.csv"
if legacy_metrics_path.is_file() and legacy_faults_path.is_file():
    legacy_metrics = pd.read_csv(legacy_metrics_path).set_index("metric")
    false_name = f"false_cases_per_{policy['exposure_unit']}"
    legacy_rate = float(legacy_metrics.at[false_name, "value"])
    nearest = comparison.iloc[(comparison.false_case_rate - legacy_rate).abs().argmin()]
    nearest_quantiles = {"rapid_residual": float(nearest.rapid_quantile)}
    if nearest.portfolio == "rapid_and_drift":
        nearest_quantiles["drift_cusum"] = float(nearest.drift_quantile)
    nearest_channels = PORTFOLIOS[nearest.portfolio]
    nearest_alerts, nearest_thresholds = make_alerts(nearest_channels, nearest_quantiles)
    nearest_cases, nearest_members = form_cases(
        nearest_alerts, entity_groups, gap_seconds=policy["case_gap_seconds"],
        thresholds=nearest_thresholds,
    )
    nearest_result = evaluate_cases(
        nearest_cases, nearest_members, fault_events, fault_intervals,
        exposure_value=exposure, exposure_unit=policy["exposure_unit"],
        decision_horizon_seconds=policy["decision_horizon_seconds"],
    )
    workload_ratio = float(nearest.false_case_rate / legacy_rate) if legacy_rate else np.nan
    legacy_summary = pd.DataFrame([
        {"version": "v2.0 frozen baseline", "event_recall": legacy_metrics.at["event_recall", "value"],
         "false_case_rate": legacy_rate, "candidate": "selected legacy configuration"},
        {"version": "v2.1 nearest workload", "event_recall": nearest.event_recall,
         "false_case_rate": nearest.false_case_rate, "candidate": nearest.candidate_key},
    ])
    legacy_summary["workload_comparable"] = abs(workload_ratio - 1) <= 0.20
    old_faults = pd.read_csv(legacy_faults_path)[
        ["fault_type", "scoreable_faults", "detected_faults", "recall"]
    ]
    new_faults = nearest_result["fault_type_results"][
        ["fault_type", "detected_faults", "recall"]
    ]
    legacy_fault_comparison = old_faults.merge(
        new_faults, on="fault_type", how="outer", suffixes=("_v2_0", "_v2_1")
    )
    display(legacy_summary)
    display(legacy_fault_comparison)
configuration = {
    "model_version": MODEL_VERSION,
    "model_core_version": MODEL_CORE_VERSION,
    "sector": SECTOR,
    "canonical_fingerprint": manifest["fingerprint"],
    "portfolio": selected.portfolio,
    "channels": selected_channels,
    "threshold_quantiles": selected_quantiles,
    "thresholds": selected_thresholds,
    "selection_status": selection_status,
    "false_case_budget": policy["false_case_budget"],
    "budget_safety_factor": policy["budget_safety_factor"],
    "budget_gate": budget_gate,
    "budget_metric": f"upper 95% confidence bound of false cases per {policy['exposure_unit']}",
    "development_exposure": exposure,
    "exposure_unit": policy["exposure_unit"],
    "case_gap_seconds": policy["case_gap_seconds"],
    "channel_persistence_seconds": policy["channel_persistence_seconds"],
    "channel_persistence_observations": persistence_observations,
    "cusum_allowance": policy["cusum_allowance"],
    "recovery_seconds": policy["recovery_seconds"],
    "recovery_observations": recovery_observations,
    "decision_horizon_seconds": policy["decision_horizon_seconds"],
    "feature_settings": decisions,
    "lookback_seconds": lookback_seconds,
    "reference_exclusions_applied": len(bundle["reference_exclusions"]),
    "candidate_count": len(comparison),
    "equivalence_margin": policy["equivalence_margin"],
    "one_fault_recall_step": policy["one_fault_recall_step"],
    "selection_evidence": policy["evidence_limit"],
    "additional_channels_status": "dispersion, PCA and Isolation Forest deferred pending incremental evidence",
    "legacy_comparison_available": not legacy_summary.empty,
    "holdout_used": False,
}

display(pd.Series(configuration, name="development_reference").to_frame())
if selection_status != "within_budget":
    print("STOP — no configuration met the development workload gate; holdout must remain sealed")
display(result["metrics"])
display(result["fault_type_results"])


## 6. Persist the small development evidence set


In [ ]:
cases = cases.sort_values("anomaly_evidence_score", ascending=False).reset_index(drop=True)
cases.insert(0, "rank", np.arange(1, len(cases) + 1))
trace = case_score_trace(
    paths["development"]["scores"],
    paths["development"]["features"],
    cases, members, alerts, bundle, selected_thresholds, selected_channels,
    window_seconds=max(
        decisions["dispersion_window_seconds"],
        20 * decisions["base_cadence_seconds"],
    ),
)

with new_output_directory(MODEL_ROOT) as output:
    joblib.dump(bundle, output / "residual_bundle.joblib")
    write_json(output / "selected_configuration.json", configuration)
    thresholds.to_csv(output / "calibration_thresholds.csv", index=False)
    comparison.to_csv(output / "development_comparison.csv", index=False)
    readiness.to_parquet(output / "development_readiness.parquet", index=False)
    alerts.to_parquet(output / "selected_alerts.parquet", index=False)
    cases.to_parquet(output / "selected_cases.parquet", index=False)
    members.to_parquet(output / "selected_case_members.parquet", index=False)
    trace.to_parquet(output / "development_case_trace.parquet", index=False)
    result["metrics"].to_csv(output / "development_metrics.csv", index=False)
    result["fault_type_results"].to_csv(output / "fault_type_results.csv", index=False)
    if not legacy_summary.empty:
        legacy_summary.to_csv(output / "legacy_workload_comparison.csv", index=False)
        legacy_fault_comparison.to_csv(output / "legacy_fault_type_comparison.csv", index=False)

print("Saved:", MODEL_ROOT)
print("Development reference portfolio:", configuration["portfolio"])
print("Next: 05_INCIDENT_RANKING_AND_HOLDOUT.ipynb")
temporary.cleanup()
